In [118]:
import pandas as pd
import numpy as np
import cv2
import os
import librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from moviepy.editor import VideoFileClip

In [119]:
num_frames = 30

In [120]:
# Step 1: Data Ingestion
def load_metadata(metadata_path):
    """Load metadata from an Excel file."""
    try:
        metadata = pd.read_excel(metadata_path)
        print(f"Metadata loaded successfully from {metadata_path}.")
        return metadata
    except Exception as e:
        print(f"Error loading metadata: {e}")
        raise


def extract_frames(video_path, num_frames):
    """Extract a specified number of frames from a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None

    frames = []
    cap = cv2.VideoCapture(video_path)
    
    while cap.isOpened() and len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        # Resize frame to a consistent size
        frame_resized = cv2.resize(frame, (64, 64))  # Resize to 64x64
        frames.append(frame_resized)

    cap.release()

    # Pad with zeros if fewer frames were extracted
    while len(frames) < num_frames:
        frames.append(np.zeros((64, 64, 3), dtype=np.uint8))  # Padding with zeros for visual features

    return np.array(frames)

def extract_visual_features(frames):
    """Extract visual features from an array of frames."""
    visual_features = []
    for frame in frames:
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        visual_features.append(gray_frame.flatten())  # Flatten the frame to a 1D array
    return np.array(visual_features)

def extract_movement_features(frames):
    """Extract movement features from an array of frames."""
    movement_features = []
    prev_frame = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    
    for i in range(1, len(frames)):
        current_frame = cv2.cvtColor(frames[i], cv2.COLOR_BGR2GRAY)
        # Calculate the absolute difference between the current frame and the previous frame
        movement = cv2.absdiff(prev_frame, current_frame)
        movement_features.append(movement.flatten())  # Flatten the movement array
        prev_frame = current_frame

    return np.array(movement_features)

def extract_speech_features(video_path, num_mfcc=13):
    """Extract speech features (MFCC) from the audio of a video file."""
    if not os.path.exists(video_path):
        print(f"Error: Video file not found: {video_path}")
        return None
    
    try:
        video = VideoFileClip(video_path)
        audio = video.audio
        
        if audio is None:
            print(f"Error: No audio found in {video_path}.")
            return None
        
        audio_array = audio.to_soundarray(fps=22000)
        # Convert to mono
        audio_mono = np.mean(audio_array, axis=1)
        # Extract MFCC features
        mfccs = librosa.feature.mfcc(y=audio_mono, sr=22000, n_mfcc=num_mfcc)
        print(f"MFCC shape: {mfccs.shape}")  # Debugging: Check shape of MFCCs
        return np.mean(mfccs.T, axis=0)  # Average MFCCs over time
    except Exception as e:
        print(f"Error processing audio for {video_path}: {e}")
        return None

In [121]:
def align_features(metadata, video_dir, num_frames):
    """Align visual, movement, and speech features and labels from metadata."""
    visual_features_list = []
    movement_features_list = []
    speech_features_list = []
    labels = []
    
    for index, row in metadata.iterrows():
        video_filename = row['VideoID']  # Assuming this column exists
        video_path = os.path.join(video_dir, video_filename)

        # Debugging: Print the video path
        print(f"Attempting to open video file: {video_path}")

        # Check if the video file exists
        if not os.path.isfile(video_path):
            print(f"Error: Could not open video file {video_path}.")
            continue  # Skip to the next file if it doesn't exist

        # Frame Extraction 
        frames = extract_frames(video_path, num_frames)

        # Extract visual features
        visual_features = extract_visual_features(frames)
        if visual_features is None:
            continue  # Skip if visual extraction failed

        # Extract movement features
        movement_features = extract_movement_features(frames)
        if movement_features is None:
            continue  # Skip if movement extraction failed

        # Extract speech features
        speech_features = extract_speech_features(video_path)
        if speech_features is None:
            continue  # Skip if speech extraction failed

        # Debugging: Check shapes of features
        print(f"Visual features shape: {visual_features.shape}")
        print(f"Movement features shape: {movement_features.shape}")
        print(f"Speech features shape: {speech_features.shape}")

        # Assuming labels are in a column named 'HowFunny'
        label = row['HowFunny']
        
        # Debugging: Check the type of the label
        print(f"Label for {video_filename}: {label} (Type: {type(label)})")
        
        visual_features_list.append(visual_features)
        movement_features_list.append(movement_features)
        speech_features_list.append(speech_features)
        labels.append(label)

    return (np.array(visual_features_list), 
            np.array(movement_features_list), 
            np.array(speech_features_list), 
            np.array(labels))

In [122]:
# Step 3: Model Training
def train_model(visual_features, movement_features, speech_features, labels):
    """Train a Random Forest model using intermediate fusion techniques."""
    # Scale features
    scaler = StandardScaler()
    visual_features_scaled = scaler.fit_transform(visual_features.reshape(-1, visual_features.shape[-1]))
    movement_features_scaled = scaler.fit_transform(movement_features.reshape(-1, movement_features.shape[-1]))
    speech_features_scaled = scaler.fit_transform(speech_features)

    # Combine features into a single array for training
    combined_features = np.concatenate((visual_features_scaled, movement_features_scaled, speech_features_scaled), axis=1)
    print(f"Combined features shape: {combined_features.shape}")  # Debugging: Check shape of combined features
    X_train, X_test, y_train, y_test = train_test_split(combined_features, labels, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    return model, X_test, y_test

In [123]:
# Step 4: Evaluation
def evaluate_model(model, X_test, y_test):
    """Evaluate the trained model."""
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

In [124]:
# Main pipeline
def main(metadata_path, video_dir, num_frames):
    metadata = load_metadata(metadata_path)
    visual_features, movement_features, speech_features, labels = align_features(metadata, video_dir, num_frames)
    
    if visual_features.size == 0 or labels.size == 0:
        print("No features or labels to train the model.")
        return
    
    model, X_test, y_test = train_model(visual_features, movement_features, speech_features, labels)
    evaluate_model(model, X_test, y_test)

# Example usage
if __name__ == "__main__":
    num_frames = 10
    main('../data/excel.xlsx', '/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata',num_frames)

Metadata loaded successfully from ../data/excel.xlsx.
Attempting to open video file: /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4
Error: Could not open video file /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep1.take1.ThatsNotAHat.mp4.
Attempting to open video file: /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4
Error: Could not open video file /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep1.take2.ThatsNotAHat.mp4.
Attempting to open video file: /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4
Error: Could not open video file /Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/NFCK3A.joke1.rep2.take1.ThatsNotAHat.mp4.
Attempting to open video file: /Users/jeremiah/video-p-reaction-joketype-modula

ValueError: all the input array dimensions for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 520 and the array at index 1 has size 468

In [ ]:
vid = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/"
vid1= "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/2RT7M4.joke1.rep1.take1.NomNomNom.mp4"
vid2 = "/Users/jeremiah/video-p-reaction-joketype-modular/Cursor_october/sdata/3ALYEd.joke4.rep2.take1.Peekaboo.mp4"

In [ ]:
# Example usage
folder_path = vid  # Specify the path to your folder of videos
num_frames = 30  # Specify the number of frames to extract
num_mfcc = 13  # Specify the number of MFCC features to extract

In [ ]:
v_f = extract_visual_features(vid)

In [ ]:
v_f

In [ ]:
m_f = extract_movement_features(vid)

In [ ]:
m_f

In [ ]:
s_f = extract_speech_features(vid)

In [ ]:
s_f.shape

In [ ]:
s_f